In [1]:
%pip -q install duckdb pyarrow

from google.colab import drive
from pathlib import Path
import os

import duckdb
import pandas as pd
import pyarrow.parquet as pq

drive.mount("/content/drive", force_remount=False)

DATA_DIR = Path("/content/drive/MyDrive/Language Detection")
PARQUET_PATH = DATA_DIR / "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
TEMP_DIR = Path("/content/duckdb_tmp")

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

TEMP_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET threads = {max(1, min(os.cpu_count() or 4, 8))}")
con.execute("SET memory_limit = '2GB'")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order = false")

parquet = pq.ParquetFile(PARQUET_PATH)
metadata = parquet.metadata

print("File :", PARQUET_PATH)
print("Size :", f"{PARQUET_PATH.stat().st_size / 1024**2:.2f} MB")
print("Rows :", f"{metadata.num_rows:,}")
print("Cols :", metadata.num_columns)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File : /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
Size : 469.49 MB
Rows : 3,469
Cols : 12


In [2]:
LANGUAGES = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt": "Portuguese",
    "es": "Spanish",
    "ru": "Russian",
}

MIN_WORDS = 4
MIN_SEGMENTS = 100
SESSIONS_PER_LANGUAGE = 5

language_sql = ", ".join(f"'{code}'" for code in LANGUAGES)

con.execute(
    f'''
    CREATE OR REPLACE TABLE session_stats AS
    SELECT
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        TRY_CAST(created_at AS TIMESTAMP) AS created_at,
        lang_detected AS language_code,
        lang_probability,
        list_count(
            list_filter(
                transcript_segments,
                segment ->
                    segment.words IS NOT NULL
                    AND len(segment.words) >= {MIN_WORDS}
            )
        ) AS segment_count
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
    WHERE
        lang_detected IN ({language_sql})
        AND transcript_segments IS NOT NULL
        AND len(transcript_segments) >= {MIN_SEGMENTS}
    '''
)

con.execute(
    f'''
    CREATE OR REPLACE TABLE selected_sessions AS
    WITH eligible AS (
        SELECT *
        FROM session_stats
        WHERE segment_count >= {MIN_SEGMENTS}
    ),
    ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY language_code
                ORDER BY
                    created_at DESC NULLS LAST,
                    segment_count ASC,
                    gamesession_id DESC
            ) AS session_rank
        FROM eligible
    )
    SELECT
        language_code,
        session_rank,
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        created_at,
        lang_probability,
        segment_count
    FROM ranked
    WHERE session_rank <= {SESSIONS_PER_LANGUAGE}
    '''
)

selected_sessions = con.execute(
    '''
    SELECT *
    FROM selected_sessions
    ORDER BY
        language_code,
        session_rank
    '''
).df()

selected_sessions.insert(
    0,
    "language",
    selected_sessions["language_code"].map(LANGUAGES),
)

display(selected_sessions)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,language,language_code,session_rank,gamesession_id,user_id,game_name,url,model_type,created_at,lang_probability,segment_count
0,German,de,1,141192575,666768,COD: Warzone3-2,https://www.twitch.tv/videos/2852368077,warfare2,2026-08-23 23:59:09,0.8784,318
1,German,de,2,141268194,838126,Star Citizen,https://www.twitch.tv/videos/2854141333,gen10,2026-08-23 23:56:20,0.9961,1139
2,German,de,3,141266889,855155,Demonologist,https://www.twitch.tv/videos/2854134239,gen5,2026-08-23 23:41:08,0.9971,1653
3,German,de,4,141268032,814284,EA Sports FC 26 & 27,https://www.twitch.tv/videos/2854265761,ea_sp_fc_26,2026-08-23 23:35:59,0.9976,415
4,German,de,5,141268217,660377,League of Legends,https://www.twitch.tv/videos/2854253024,lol,2026-08-23 23:35:07,0.9683,295
5,English,en,1,141268501,424509,Quarantine Zone: The Last Check,https://www.twitch.tv/videos/2854311727,gen10,2026-08-23 23:59:46,0.9863,431
6,English,en,2,141268683,805510,COD: Warzone3-2,https://www.twitch.tv/videos/2854243785,warfare2,2026-08-23 23:59:31,0.9961,280
7,English,en,3,141267410,185094,Dune: Awakening,https://www.twitch.tv/videos/2854174984,gen10,2026-08-23 23:59:20,0.9985,1533
8,English,en,4,141268467,386936,COD: Modern Warfare III,https://www.twitch.tv/videos/2854318859,gen10,2026-08-23 23:57:01,0.9990,196
9,English,en,5,141268071,856479,Phasmophobia,https://www.twitch.tv/videos/2853103396,gen10,2026-08-23 23:56:22,0.9985,403


In [3]:
con.execute(
    f'''
    CREATE OR REPLACE TABLE selected_segments AS
    WITH source AS (
        SELECT
            p.gamesession_id,
            p.lang_detected AS language_code,
            p.transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}') AS p
        INNER JOIN selected_sessions AS s
            ON p.gamesession_id = s.gamesession_id
            AND p.lang_detected = s.language_code
    ),
    exploded AS (
        SELECT
            gamesession_id,
            language_code,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM source
    )
    SELECT
        gamesession_id,
        language_code,
        segment_index,
        TRIM(segment.text) AS segment_text,
        len(segment.words) AS word_count
    FROM exploded
    WHERE
        segment.words IS NOT NULL
        AND len(segment.words) >= {MIN_WORDS}
        AND segment.text IS NOT NULL
        AND TRIM(segment.text) <> ''
    '''
)

selected_segment_counts = con.execute(
    '''
    SELECT
        language_code,
        gamesession_id,
        COUNT(*) AS selected_segment_count,
        MIN(word_count) AS min_word_count
    FROM selected_segments
    GROUP BY
        language_code,
        gamesession_id
    ORDER BY
        language_code,
        gamesession_id
    '''
).df()

display(selected_segment_counts)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,language_code,gamesession_id,selected_segment_count,min_word_count
0,de,141192575,318,4
1,de,141266889,1653,4
2,de,141268032,415,4
3,de,141268194,1139,4
4,de,141268217,295,4
5,en,141267410,1533,4
6,en,141268071,403,4
7,en,141268467,196,4
8,en,141268501,431,4
9,en,141268683,280,4


In [4]:
session_validation = con.execute(
    f'''
    WITH language_checks AS (
        SELECT
            language_code,
            COUNT(*) AS session_count,
            MIN(segment_count) AS min_segment_count,
            COUNT(*) = {SESSIONS_PER_LANGUAGE} AS has_five_sessions,
            MIN(segment_count) >= {MIN_SEGMENTS} AS all_sessions_have_100_segments
        FROM selected_sessions
        GROUP BY language_code
    ),
    word_checks AS (
        SELECT
            language_code,
            MIN(word_count) AS min_word_count,
            MIN(word_count) >= {MIN_WORDS} AS all_segments_have_four_words
        FROM selected_segments
        GROUP BY language_code
    ),
    order_checks AS (
        SELECT
            language_code,
            bool_and(
                next_created_at IS NULL
                OR created_at >= next_created_at
            ) AS newest_first
        FROM (
            SELECT
                language_code,
                session_rank,
                created_at,
                lead(created_at) OVER (
                    PARTITION BY language_code
                    ORDER BY session_rank
                ) AS next_created_at
            FROM selected_sessions
        )
        GROUP BY language_code
    )
    SELECT
        l.language_code,
        l.session_count,
        l.min_segment_count,
        w.min_word_count,
        l.has_five_sessions,
        l.all_sessions_have_100_segments,
        w.all_segments_have_four_words,
        o.newest_first,
        (
            l.has_five_sessions
            AND l.all_sessions_have_100_segments
            AND w.all_segments_have_four_words
            AND o.newest_first
        ) AS all_checks_passed
    FROM language_checks AS l
    INNER JOIN word_checks AS w USING (language_code)
    INNER JOIN order_checks AS o USING (language_code)
    ORDER BY language_code
    '''
).df()

session_validation.insert(
    0,
    "language",
    session_validation["language_code"].map(LANGUAGES),
)

display(session_validation)

if len(session_validation) != len(LANGUAGES):
    raise ValueError("Validation did not cover all target languages")

if not session_validation["all_checks_passed"].all():
    raise ValueError("Validation failed")

print("All validation checks passed.")


,language,language_code,session_count,min_segment_count,min_word_count,has_five_sessions,all_sessions_have_100_segments,all_segments_have_four_words,newest_first,all_checks_passed
0,German,de,5,295,4,True,True,True,True,True
1,English,en,5,196,4,True,True,True,True,True
2,Spanish,es,5,111,4,True,True,True,True,True
3,French,fr,5,224,4,True,True,True,True,True
4,Portuguese,pt,5,159,4,True,True,True,True,True
5,Russian,ru,5,157,4,True,True,True,True,True


All validation checks passed.
